# 01 数据预处理

## 目标
- 加载阿里天池 UserBehavior 数据集
- 数据清洗：去重、缺失值处理、时间格式转换
- 数据采样：原始数据约1亿条，按需采样
- 导出清洗后的数据供后续分析使用

## 数据字段说明
| 字段 | 说明 |
|------|------|
| user_id | 用户ID |
| item_id | 商品ID |
| category_id | 商品类目ID |
| behavior_type | 行为类型：pv(浏览), fav(收藏), cart(加购), buy(购买) |
| timestamp | Unix时间戳（秒） |

In [2]:
# 导入必要的库
import pandas as pd
import numpy as np
import os
import sys

# 添加项目根目录到路径，方便导入工具函数
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

# 设置显示选项
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('库加载完成')

库加载完成


## 1. 加载数据

In [3]:
# 数据文件路径（请根据实际路径修改）
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'UserBehavior.csv')

# 定义列名（原始CSV无表头）
col_names = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']

# 加载数据
# 注意：原始数据约1亿条，首次加载可能需要几分钟
df_raw = pd.read_csv(DATA_PATH, names=col_names, header=0)

print(f'数据加载完成，共 {len(df_raw):,} 条记录')
print(f'数据形状: {df_raw.shape}')
df_raw.head()

数据加载完成，共 100,150,806 条记录
数据形状: (100150806, 5)


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2333346,2520771,pv,1511561733
1,1,2576651,149192,pv,1511572885
2,1,3830808,4181361,pv,1511593493
3,1,4365585,2520377,pv,1511596146
4,1,4606018,2735466,pv,1511616481


## 2. 数据概览

In [4]:
# 基本信息
print('=== 数据基本信息 ===')
print(df_raw.info())
print()
print('=== 数据统计描述 ===')
print(df_raw.describe())

=== 数据基本信息 ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100150806 entries, 0 to 100150805
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   user_id        int64 
 1   item_id        int64 
 2   category_id    int64 
 3   behavior_type  object
 4   timestamp      int64 
dtypes: int64(4), object(1)
memory usage: 3.7+ GB
None

=== 数据统计描述 ===
           user_id      item_id  category_id      timestamp
count 100150806.00 100150806.00 100150806.00   100150806.00
mean     506943.13   2579774.80   2696380.39  1511951276.88
std      294060.50   1488055.99   1463154.53     5528006.10
min           1.00         1.00        80.00 -2134949234.00
25%      252429.00   1295225.00   1320293.00  1511761843.00
50%      504015.00   2580735.00   2671397.00  1511964596.00
75%      760949.00   3862042.00   4145813.00  1512179082.00
max     1018011.00   5163070.00   5162429.00  2122867355.00


In [5]:
# 唯一值统计
print('=== 唯一值统计 ===')
print(f'用户数量: {df_raw["user_id"].nunique():,}')
print(f'商品数量: {df_raw["item_id"].nunique():,}')
print(f'类目数量: {df_raw["category_id"].nunique():,}')
print(f'行为类型: {df_raw["behavior_type"].unique()}')
print()

# 行为类型分布
print('=== 行为类型分布 ===')
behavior_counts = df_raw['behavior_type'].value_counts()
behavior_map = {'pv': '浏览', 'fav': '收藏', 'cart': '加购', 'buy': '购买'}
for bt, count in behavior_counts.items():
    pct = count / len(df_raw) * 100
    print(f'{behavior_map.get(bt, bt)} ({bt}): {count:,} ({pct:.2f}%)')

=== 唯一值统计 ===
用户数量: 987,994
商品数量: 4,162,024
类目数量: 9,439
行为类型: ['pv' 'fav' 'buy' 'cart']

=== 行为类型分布 ===
浏览 (pv): 89,716,263 (89.58%)
加购 (cart): 5,530,446 (5.52%)
收藏 (fav): 2,888,258 (2.88%)
购买 (buy): 2,015,839 (2.01%)


## 3. 数据清洗

In [6]:
# 3.1 检查缺失值
print('=== 缺失值检查 ===')
missing = df_raw.isnull().sum()
print(missing)
print(f'\n总缺失值: {missing.sum()}')

=== 缺失值检查 ===
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64

总缺失值: 0


In [7]:
# 3.2 检查重复值
dup_count = df_raw.duplicated().sum()
print(f'重复记录数: {dup_count:,}')
print(f'重复比例: {dup_count/len(df_raw)*100:.2f}%')

# 去重
df_clean = df_raw.drop_duplicates()
print(f'\n去重后记录数: {len(df_clean):,}')
print(f'去除记录数: {len(df_raw) - len(df_clean):,}')

重复记录数: 49
重复比例: 0.00%

去重后记录数: 100,150,757
去除记录数: 49


In [10]:
# 3.3 时间戳转换
df_clean['datetime'] = pd.to_datetime(df_clean['timestamp'], unit='s')
df_clean['date'] = df_clean['datetime'].dt.date
df_clean['hour'] = df_clean['datetime'].dt.hour
df_clean['weekday'] = df_clean['datetime'].dt.weekday  # 0=周一, 6=周日
df_clean['weekday_name'] = df_clean['datetime'].dt.day_name()

print('时间范围:')
print(f'  最早: {df_clean["datetime"].min()}')
print(f'  最晚: {df_clean["datetime"].max()}')
print(f'  天数: {(df_clean["datetime"].max() - df_clean["datetime"].min()).days} 天')

df_clean.head()

c:\Users\PC\AppData\Local\Programs\Python\Python37\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
c:\Users\PC\AppData\Local\Programs\Python\Python37\lib\site-packages\ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until
c:\Users\PC\AppData\Local\Programs\Python\Python37\lib\site-packages\ipykernel_launcher.py:4: SettingWithCopyWarning: 
A value is trying to be set on a

时间范围:
  最早: 1902-05-07 22:32:46
  最晚: 2037-04-09 05:22:35
  天数: 49280 天


,user_id,item_id,category_id,behavior_type,timestamp,datetime,date,hour,weekday,weekday_name
0,1,2333346,2520771,pv,1511561733,2017-11-24 22:15:33,2017-11-24,22,4,Friday
1,1,2576651,149192,pv,1511572885,2017-11-25 01:21:25,2017-11-25,1,5,Saturday
2,1,3830808,4181361,pv,1511593493,2017-11-25 07:04:53,2017-11-25,7,5,Saturday
3,1,4365585,2520377,pv,1511596146,2017-11-25 07:49:06,2017-11-25,7,5,Saturday
4,1,4606018,2735466,pv,1511616481,2017-11-25 13:28:01,2017-11-25,13,5,Saturday


## 4. 数据采样

原始数据约1亿条，为了提高后续分析效率，我们进行采样。
采样策略：随机抽取一定数量的用户，保留这些用户的全部行为记录。

In [11]:
# 采样策略：抽取 50000 个用户的所有行为记录
SAMPLE_USERS = 50000

# 随机选择用户
np.random.seed(42)
sampled_users = np.random.choice(
    df_clean['user_id'].unique(),
    size=min(SAMPLE_USERS, df_clean['user_id'].nunique()),
    replace=False
)

# 筛选这些用户的全部记录
df_sample = df_clean[df_clean['user_id'].isin(sampled_users)].copy()

print(f'采样用户数: {len(sampled_users):,}')
print(f'采样记录数: {len(df_sample):,}')
print(f'采样比例: {len(df_sample)/len(df_clean)*100:.2f}%')
print()
print('采样后行为分布:')
behavior_map = {'pv': '浏览', 'fav': '收藏', 'cart': '加购', 'buy': '购买'}
for bt, count in df_sample['behavior_type'].value_counts().items():
    pct = count / len(df_sample) * 100
    print(f'  {behavior_map.get(bt, bt)}: {count:,} ({pct:.2f}%)')

采样用户数: 50,000
采样记录数: 5,070,824
采样比例: 5.06%

采样后行为分布:
  浏览: 4,542,940 (89.59%)
  加购: 278,373 (5.49%)
  收藏: 146,383 (2.89%)
  购买: 103,128 (2.03%)


## 5. 数据验证

In [12]:
# 验证采样数据的分布是否与原始数据一致
print('=== 原始数据 vs 采样数据 行为分布对比 ===')
print()

original_dist = df_clean['behavior_type'].value_counts(normalize=True) * 100
sample_dist = df_sample['behavior_type'].value_counts(normalize=True) * 100

comparison = pd.DataFrame({
    '原始数据(%)': original_dist,
    '采样数据(%)': sample_dist
}).round(2)

print(comparison)

=== 原始数据 vs 采样数据 行为分布对比 ===

      原始数据(%)  采样数据(%)
pv      89.58    89.59
cart     5.52     5.49
fav      2.88     2.89
buy      2.01     2.03


## 6. 导出清洗后的数据

In [13]:
# 导出采样数据
OUTPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'user_behavior_clean.csv')
df_sample.to_csv(OUTPUT_PATH, index=False)
print(f'清洗后数据已保存到: {OUTPUT_PATH}')
print(f'文件大小: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.2f} MB')

清洗后数据已保存到: c:\Users\PC\ecommerce-user-behavior-analysis\data\user_behavior_clean.csv
文件大小: 393.14 MB


In [14]:
# 也导出一份用于 SQLite 的数据（方便后续 SQL 分析）
import sqlite3

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'user_behavior.db')
conn = sqlite3.connect(DB_PATH)

# 写入 SQLite
df_sample.to_sql('user_behavior', conn, if_exists='replace', index=False)
print(f'SQLite 数据库已保存到: {DB_PATH}')

# 验证
result = pd.read_sql('SELECT COUNT(*) as cnt FROM user_behavior', conn)
print(f'数据库记录数: {result["cnt"].values[0]:,}')

conn.close()

SQLite 数据库已保存到: c:\Users\PC\ecommerce-user-behavior-analysis\data\user_behavior.db
数据库记录数: 5,070,824


## 7. 预处理总结

完成的预处理步骤：
1. **数据加载**：加载原始 CSV 数据
2. **缺失值检查**：确认无缺失值
3. **去重处理**：去除重复记录
4. **时间转换**：Unix时间戳转为可读时间，提取日期、小时、星期等特征
5. **数据采样**：抽取5万用户的行为记录，保留完整用户行为链
6. **数据导出**：保存为 CSV 和 SQLite 两种格式，方便后续分析

### 后续分析使用的数据文件
- `data/user_behavior_clean.csv` - 清洗后的采样数据（Python分析用）
- `data/user_behavior.db` - SQLite数据库（SQL分析用）